# 📝 Prompt Templates

## What is a Prompt Template?
A **template** is a reusable structure with **variables** (placeholders).

Instead of writing: `"Translate 'Hello' to Spanish"`
You write: `"Translate '{text}' to {language}"`

Then fill in variables at runtime.

## Mental Model: Mad Libs
Remember Mad Libs? Fill in blanks with words:
```
"The {adjective} {noun} {verb} over the {noun}."
```
PromptTemplate = Professional Mad Libs for AI prompts.

## Why Not Just f-strings?
Python f-strings work for simple cases. PromptTemplates add:
- **Type validation** — catches missing variables early
- **Composability** — templates work inside LCEL chains
- **Serialization** — save/load templates from files
- **Partial fills** — fill some variables now, rest later
- **Structured output** — ChatPromptTemplate creates proper Messages


In [ ]:
# ── PromptTemplate vs f-string ─────────────────────────────────────────
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# ── f-string approach (naive) ──
def naive_prompt(product: str, language: str) -> str:
    return f'Write a product description for {product} in {language}.'
# Problem: just a string, can't be used in LCEL chains natively

# ── PromptTemplate approach ──
template = PromptTemplate(
    input_variables=['product', 'language'],  # declared variables
    template='Write a product description for {product} in {language}.'
)

# Shorthand (auto-detects variables from template string)
template2 = PromptTemplate.from_template(
    'Write a product description for {product} in {language}.'
)

# Fill in variables
result = template.invoke({'product': 'Mechanical Keyboard', 'language': 'French'})
print('Type:', type(result).__name__)  # StringPromptValue
print('Text:', result.text)
print('Input variables:', template.input_variables)

In [ ]:
# ── ChatPromptTemplate ─────────────────────────────────────────────────
# WHY: Chat LLMs need MESSAGES not strings.
# ChatPromptTemplate builds a list of messages from a template.

from langchain_core.prompts import ChatPromptTemplate

# from_messages takes a list of (role, template) tuples
chat_template = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert {domain} tutor. Be concise and clear.'),
    ('human', 'Explain {concept} with a practical example.'),
])

# Fill variables
result = chat_template.invoke({'domain': 'Python', 'concept': 'closures'})
print('Type:', type(result).__name__)  # ChatPromptValue
print('Messages:')
for msg in result.messages:
    print(f'  [{msg.type}]: {msg.content}')

In [ ]:
# ── Partial Templates: Fill Variables in Stages ────────────────────────
# WHY: Sometimes you know some variables at startup, others at runtime.
# Example: language is fixed at startup, user_input changes per request.

from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ('system', 'You translate text to {language}. Be accurate.'),
    ('human', '{text}'),
])

# Partial fill: set 'language' now
spanish_translator = template.partial(language='Spanish')

# At runtime: only need to provide 'text'
result1 = spanish_translator.invoke({'text': 'Hello, how are you?'})
result2 = spanish_translator.invoke({'text': 'Good morning!'})

print('Template 1 messages:')
for m in result1.messages: print(f'  {m.type}: {m.content}')

print('\nTemplate 2 messages:')
for m in result2.messages: print(f'  {m.type}: {m.content}')

In [ ]:
# ── MessagesPlaceholder: Dynamic Chat History ───────────────────────────
# WHY: In chatbots, you need to inject conversation history into prompt.
# MessagesPlaceholder is a slot where you inject a list of messages.

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

template = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    MessagesPlaceholder(variable_name='history'),  # <-- dynamic slot!
    ('human', '{input}'),
])

# Simulate a conversation history
chat_history = [
    HumanMessage(content='My name is Alice.'),
    AIMessage(content='Nice to meet you, Alice!'),
]

result = template.invoke({
    'history': chat_history,
    'input': 'What is my name?'
})

print('Full message list sent to LLM:')
for msg in result.messages:
    print(f'  [{msg.type}]: {msg.content}')

## 🎯 Interview Questions

1. **What is the difference between PromptTemplate and ChatPromptTemplate?**
2. **What is partial() used for in prompt templates?**
3. **What is MessagesPlaceholder and why is it needed?**
4. **Why use PromptTemplate instead of Python f-strings?**
5. **How do you serialize a PromptTemplate to disk?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Prompt Templates — Reusable, Structured Prompts**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
